# The HS-AFM Lab: Watching Proteins Under a Virtual Microscope

**High-Speed Atomic Force Microscopy (HS-AFM)** is the only imaging technique that watches
individual protein molecules move in real time, in solution, at nanometer resolution. It has
revealed how myosin "walks" along actin filaments, how chaperones grab unfolded clients,
and how intrinsically disordered proteins explore conformational space.

But interpreting HS-AFM movies is hard — the images are blurred by the **tip geometry**,
distorted by **scanning lag** (the tip moves line-by-line while the protein moves), and
corrupted by thermal noise.

**`synth-afm`** simulates every one of these effects from atomic coordinates, letting you
build the physical intuition needed to interpret real experiments.

> **In this lab** we generate synthetic HS-AFM images for three protein architectures,
> compare tip-dilation effects, and create a frame-by-frame movie of an ENM trajectory.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q synth-pdb synth-afm synth-dynamics biotite matplotlib numpy jax MDAnalysis
else:
    sys.path.insert(0, os.path.abspath("../../"))

import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
from IPython.display import HTML
import jax.numpy as jnp
import biotite.structure.io.pdb as bpdb
from synth_pdb.generator import generate_pdb_content
from synth_afm.simulator import AFMSimulator
from synth_afm.io import VDW_RADII, DEFAULT_ATOM_RADIUS

print("✅ Environment ready")

## How HS-AFM Works: Tip Meets Molecule

An AFM probe is a sharp needle (tip radius ~1–10 nm) mounted on a cantilever spring.
As the tip scans in a raster pattern over the surface:

1. **The tip deflects** when it encounters the protein surface
2. **Deflection is measured** by a laser reflecting off the cantilever
3. **Height is recorded** pixel by pixel — producing a topographic height map

Two physical effects corrupt the idealized image:

- **Tip dilation**: The finite tip radius "broadens" all features by up to the tip diameter.
  A 10 nm tip adds ~10 nm to the apparent lateral size of any feature.
- **Scanning lag**: The tip scans line-by-line (top to bottom). If the protein moves during
  a scan (~10–100 ms), different rows of the image capture the protein at different moments
  — producing a characteristic "shearing" artifact.

Both effects are modelled physically by `synth-afm`.

In [ ]:
def prepare_coords(struct, grid_pixels=64, margin_ang=12.0):
    """Centre & shift coordinates so the protein fits in the AFM grid."""
    coords = np.array(struct.coord, dtype=np.float32)
    # Centre xy, zero z
    coords[:, 0] -= coords[:, 0].mean()
    coords[:, 1] -= coords[:, 1].mean()
    coords[:, 2] -= coords[:, 2].min()
    # Choose pixel size to fit protein + margin
    span = max(coords[:, 0].max() - coords[:, 0].min(), coords[:, 1].max() - coords[:, 1].min())
    pixel_size = (span + 2 * margin_ang) / grid_pixels
    # Shift to positive quadrant (leave margin)
    coords[:, 0] += span / 2 + margin_ang
    coords[:, 1] += span / 2 + margin_ang
    return coords, pixel_size


def get_radii(struct):
    return np.array(
        [VDW_RADII.get(e.upper(), DEFAULT_ATOM_RADIUS) for e in struct.element], dtype=np.float32
    )


def parse_pdb(pdb_str):
    return bpdb.PDBFile.read(io.StringIO(pdb_str)).get_structure(model=1)


print("Helper functions ready.")

## Three Protein Architectures Under the Virtual Tip

We'll image three fundamentally different proteins:

| Structure | Architecture | Expected height map |
|---|---|---|
| **Compact helix bundle** | Dense α-helical coil | Tall, compact blob |
| **Extended β-sheet** | Flat ribbon | Flatter, elongated shape |
| **Intrinsically disordered** | GS-linker (IDP) | Low, spread-out texture |

The contrast difference between a folded globular protein and an IDP is immediately
visible in HS-AFM — and directly diagnostic.

In [ ]:
GRID = 64
print("Generating protein structures...")

architectures = [
    ("Compact Helix", "AAAAAKKKKAAAAAKKKKAAAAAKKKKAAAA", "alpha"),
    ("Extended Sheet", "VVVVVVVVVVVVVVVVVVVV", "beta"),
    ("Disordered IDP", "GSGSGSGSGSGSGSGSGSGSGSGSGSGSGS", "random"),
]

structs, hmaps, pixel_sizes = {}, {}, {}

for name, seq, conf in architectures:
    pdb_str = generate_pdb_content(sequence_str=seq, conformation=conf, seed=42)
    struct = parse_pdb(pdb_str)
    coords, ps = prepare_coords(struct, GRID)
    radii = get_radii(struct)
    sim = AFMSimulator(pixel_size=ps, grid_size=(GRID, GRID), tip_radius=12.0)
    hmap = np.array(sim.scan(jnp.array(coords), jnp.array(radii)))
    structs[name] = struct
    hmaps[name] = hmap
    pixel_sizes[name] = ps
    z_max = hmap.max()
    print(f"  {name:20s}: pixel={ps:.2f}Å  max_height={z_max:.2f}Å")

print("\nAll height maps computed.")

## The AFM Gallery: Three Proteins, Three Signatures

The **plasma** colormap maps low height to dark purple and maximum height to bright yellow-white
— the same convention used in real HS-AFM publications. The height scale bar shows the
topographic range in Ångströms.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor("#0d1117")

for ax, (name, seq, conf) in zip(axes, architectures):
    hmap = hmaps[name]
    ps = pixel_sizes[name]
    extent = [0, GRID * ps, 0, GRID * ps]
    im = ax.imshow(hmap.T, origin="lower", cmap="plasma", extent=extent, interpolation="bilinear")
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Height (Å)", color="#cdd6f4", fontsize=9)
    cbar.ax.yaxis.set_tick_params(color="#cdd6f4")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#cdd6f4")

    ax.set_title(name, color="#cdd6f4", fontsize=13, fontweight="bold", pad=8)
    ax.set_xlabel("x (Å)", color="#8fa3bf", fontsize=10)
    ax.set_ylabel("y (Å)", color="#8fa3bf", fontsize=10)
    ax.tick_params(colors="#8fa3bf")
    ax.set_facecolor("#0d1117")
    for spine in ax.spines.values():
        spine.set_edgecolor("#313244")

fig.suptitle(
    "Synthetic HS-AFM Images — tip radius 12 Å",
    color="#cdd6f4",
    fontsize=15,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig("afm_gallery.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("🔬 Compact helix = tallest peak | IDP = lowest, diffuse texture")

## The Tip Dilation Effect: How Sharp Is Your Tip?

A finer tip (smaller radius) gives sharper lateral resolution but is more fragile and more
likely to damage the sample. The **Villarrubia dilation formula** predicts how a spherical
tip of radius R broadens a point feature by ~2R laterally.

Compare the same helix imaged with three different tip radii — this is what changes when
you choose your cantilever at the start of an experiment.

In [ ]:
tip_radii = [5.0, 12.0, 30.0]  # Å  (sharp, standard, blunt)
tip_labels = ["Sharp (5 Å)", "Standard (12 Å)", "Blunt (30 Å)"]

helix_pdb = generate_pdb_content(
    sequence_str="AAAAAKKKKAAAAAKKKKAAAAAKKKKAAAA", conformation="alpha", seed=42
)
helix_struct = parse_pdb(helix_pdb)
coords_h, ps_h = prepare_coords(helix_struct, GRID)
radii_h = get_radii(helix_struct)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor("#0d1117")

for ax, tip_r, label in zip(axes, tip_radii, tip_labels):
    sim = AFMSimulator(pixel_size=ps_h, grid_size=(GRID, GRID), tip_radius=tip_r)
    hmap = np.array(sim.scan(jnp.array(coords_h), jnp.array(radii_h)))
    extent = [0, GRID * ps_h, 0, GRID * ps_h]
    im = ax.imshow(hmap.T, origin="lower", cmap="hot", extent=extent, interpolation="bilinear")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(
        "Height (Å)", color="#cdd6f4", fontsize=9
    )
    ax.set_title(label, color="#cdd6f4", fontsize=13, fontweight="bold", pad=8)
    ax.set_facecolor("#0d1117")
    ax.tick_params(colors="#8fa3bf")
    for spine in ax.spines.values():
        spine.set_edgecolor("#313244")

fig.suptitle(
    "Tip Dilation Effect on an α-Helix Bundle\nFiner tip → sharper lateral features",
    color="#cdd6f4",
    fontsize=14,
    fontweight="bold",
    y=1.04,
)
plt.tight_layout()
plt.savefig("afm_tip_comparison.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

## The HS-AFM Movie: Capturing Motion Frame by Frame

The defining power of **high-speed** AFM is temporal resolution — modern instruments
achieve 10–1000 frames per second, fast enough to watch enzymes at work.

We'll simulate an ENM Langevin dynamics trajectory and feed it directly to the AFM
simulator to generate a synthetic HS-AFM movie. Each frame represents one snapshot
of the protein's thermal fluctuations.

> **Note**: The scanning-lag artifact is automatically included: different rows of each
> frame are sampled from slightly different time points in the trajectory, producing
> the characteristic shearing seen in real HS-AFM data.

In [ ]:
import tempfile
import MDAnalysis as mda  # noqa: N813
from synth_dynamics import ANMForceField, LangevinIntegrator, Simulation, System

# ── Generate ubiquitin and run ENM Langevin dynamics ──
UBQ_SEQ = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
ubq_pdb = generate_pdb_content(sequence_str=UBQ_SEQ, conformation="alpha", seed=42)

with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False, mode="w") as f:
    f.write(ubq_pdb)
    tmp_pdb = f.name

sys_d = System(tmp_pdb)
ff_d = ANMForceField(sys_d.equilibrium_coords, cutoff=13.0, spring_constant=0.8)
integrator = LangevinIntegrator(dt=0.05, temperature=300.0, friction=0.8)
sim_d = Simulation(sys_d, ff_d, integrator)

traj_path = tempfile.mktemp(suffix="_ubq.pdb")
N_STEPS = 160
STRIDE = 8  # 20 frames
sim_d.run(n_steps=N_STEPS, output_path=traj_path, stride=STRIDE)

# ── Read CA trajectory ──
u = mda.Universe(traj_path)
ca_frames = []
for ts in u.trajectory:
    ca_frames.append(u.atoms.positions.copy())
ca_traj = np.array(ca_frames, dtype=np.float32)  # (T, N_CA, 3)
T, N_CA, _ = ca_traj.shape
print(f"CA trajectory: {T} frames, {N_CA} Cα atoms")

# ── Prepare trajectory for AFM (centre each frame) ──
# Use frame-0 to compute stable pixel size
f0 = ca_traj[0].copy()
f0[:, 0] -= f0[:, 0].mean()
f0[:, 1] -= f0[:, 1].mean()
f0[:, 2] -= f0[:, 2].min()
span = max(f0[:, 0].max() - f0[:, 0].min(), f0[:, 1].max() - f0[:, 1].min())
MOV_GRID = 48
ps_mov = (span + 25) / MOV_GRID
shift_x = span / 2 + 12
shift_y = span / 2 + 12

traj_prepared = np.zeros((T, N_CA, 3), dtype=np.float32)
for t in range(T):
    fr = ca_traj[t].copy()
    fr[:, 0] -= ca_traj[0, :, 0].mean()
    fr[:, 1] -= ca_traj[0, :, 1].mean()
    fr[:, 2] -= ca_traj[0, :, 2].min()
    fr[:, 0] += shift_x
    fr[:, 1] += shift_y
    traj_prepared[t] = fr

ca_radii = np.full(N_CA, 1.7, dtype=np.float32)  # Carbon vdW radius

# ── Run AFM scan_movie ──
sim_afm = AFMSimulator(pixel_size=ps_mov, grid_size=(MOV_GRID, MOV_GRID), tip_radius=10.0)
movie_frames = np.array(
    sim_afm.scan_movie(
        jnp.array(traj_prepared),
        radii=jnp.array(ca_radii),
        frames_per_second=10.0,
    )
)  # (T, H, W)
print(f"AFM movie: {movie_frames.shape} (frames, H, W)")
print(f"Height range across movie: {movie_frames.min():.2f} – {movie_frames.max():.2f} Å")

import os

os.unlink(tmp_pdb)
os.unlink(traj_path)

In [ ]:
# Render AFM movie as matplotlib animation
vmin = movie_frames.min()
vmax = movie_frames.max()
extent = [0, MOV_GRID * ps_mov, 0, MOV_GRID * ps_mov]

fig_m, ax_m = plt.subplots(figsize=(5.5, 5.5))
fig_m.patch.set_facecolor("#0d1117")
ax_m.set_facecolor("#0d1117")
for spine in ax_m.spines.values():
    spine.set_edgecolor("#313244")

im_m = ax_m.imshow(
    movie_frames[0].T,
    origin="lower",
    cmap="plasma",
    extent=extent,
    interpolation="bilinear",
    vmin=vmin,
    vmax=vmax,
)
cbar_m = plt.colorbar(im_m, ax=ax_m, fraction=0.046, pad=0.04)
cbar_m.set_label("Height (Å)", color="#cdd6f4", fontsize=9)
cbar_m.ax.yaxis.set_tick_params(color="#cdd6f4")
plt.setp(cbar_m.ax.yaxis.get_ticklabels(), color="#cdd6f4")

title_m = ax_m.set_title("HS-AFM Frame 0", color="#cdd6f4", fontsize=13, fontweight="bold")
ax_m.set_xlabel("x (Å)", color="#8fa3bf")
ax_m.set_ylabel("y (Å)", color="#8fa3bf")
ax_m.tick_params(colors="#8fa3bf")
plt.tight_layout()


def _update(frame_idx):
    im_m.set_data(movie_frames[frame_idx].T)
    title_m.set_text(f"HS-AFM Frame {frame_idx + 1}/{T} — Ubiquitin ENM dynamics")
    return im_m, title_m


ani = animation.FuncAnimation(fig_m, _update, frames=T, interval=150, blit=True)
plt.close(fig_m)  # prevent static display
HTML(ani.to_jshtml())

## What We've Demonstrated

| Experiment | Scientific insight |
|---|---|
| Three-architecture gallery | Shape differences are immediately visible — folded vs disordered |
| Tip dilation comparison | Lateral resolution is fundamentally limited by tip geometry |
| ENM dynamics movie | Thermal fluctuations create a "breathing" motion visible in HS-AFM |

**Real-world applications of HS-AFM** that `synth-afm` can help train models for:

- **Protein-DNA interactions**: Watching transcription factors bind and release
- **Chaperone dynamics**: GroEL opening and closing its cavity in real time  
- **IDP aggregation**: Observing the early nucleation events in amyloid formation
  (*α*-synuclein, tau, amyloid-β)
- **Motor proteins**: Myosin V taking 36 nm steps along actin — directly imaged!

The movies generated here are ground-truth synthetic data — exactly what's needed
to train the denoising and state-detection neural networks that will make HS-AFM
interpretation fully automatic.